# Exploring Chunking Strategies and Vector Databases

In this notebook, you will explore and compare chunking strategies for Retrieval-Augmented Generation (RAG), then evaluate how those choices influence retrieval outcomes.

Chunking is the process of splitting a document into smaller pieces (chunks). Each chunk can then be embedded, indexed, and retrieved. Strong chunking choices often improve retrieval relevance and answer quality in RAG systems.

This notebook combines step-by-step experiments with interactive dashboards so you can inspect chunks and retrieval results more efficiently.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/01.rag-chunking.ipynb)  

## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Understand why chunking strategy matters in RAG.
- Compare simple, recursive, semantic, and LLM-based chunking methods on the same source text.
- Build vector databases from each chunking output and inspect retrieval behavior.
- Use interactive dashboards to compare chunk content side-by-side and explore similarity search results by strategy.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [1]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

In [2]:
from aup_config import aup_setup
aup_setup()

## Let's Chunk

### Import Libraries

In [3]:
!pip install -U langchain-ollama


In [4]:
import os
import requests
import re
import ipywidgets as widgets
!pip install -U langchain-ollama
from langchain_ollama import ChatOllama
from IPython.display import display

import langchain_core

from langchain_text_splitters import (
    TokenTextSplitter,
    RecursiveCharacterTextSplitter
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker

from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_114/709973966.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_114/709973966.py:17: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [5]:
llm = ChatOllama(
    model="Llama3.1:8b",
    base_url="http://localhost:11434"
)

### Create Embedding Model


In [6]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

### Create text file

In [7]:
medicine_text = """
Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day.
Avoid alcohol while taking this medicine.

Medicine: Aspirin

Uses:
Pain relief
Reduces fever
Blood thinner

Dosage:
75–325 mg as prescribed.

Side Effects:
Stomach irritation
Bleeding
Heartburn

Precautions:
Avoid if allergic to aspirin.
"""

with open("medicine_data/medicine_info.txt", "w") as f:
    f.write(medicine_text)

print("medicine_info.txt created successfully!")

medicine_info.txt created successfully!


### Load it

In [8]:
from langchain_core.documents import Document

with open("medicine_data/medicine_info.txt", "r", encoding="utf-8") as f:
    text = f.read()

docs_merged = [Document(page_content=text)]

print("Characters:", len(text))
print(text[:200])

Characters: 450

Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 gram


Also check whether docs was overwritten

In [9]:
docs_merged

[Document(metadata={}, page_content='\nMedicine: Paracetamol\n\nUses:\nRelieves fever and mild to moderate pain.\n\nDosage:\n500 mg every 4–6 hours.\n\nSide Effects:\nNausea\nVomiting\nRash\nLiver damage in overdose\n\nPrecautions:\nDo not exceed 4 grams per day.\nAvoid alcohol while taking this medicine.\n\nMedicine: Aspirin\n\nUses:\nPain relief\nReduces fever\nBlood thinner\n\nDosage:\n75–325 mg as prescribed.\n\nSide Effects:\nStomach irritation\nBleeding\nHeartburn\n\nPrecautions:\nAvoid if allergic to aspirin.\n')]

Then combine the text:

In [10]:
text = ""

for doc in docs_merged:
    text += doc.page_content

print("Characters:", len(text))
print(text[:300]) 

Characters: 450

Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day.
Avoid alcohol while taking this medicine.

Medicine: Aspirin

Uses:
Pain relief
Reduces f


## Chunking Strategies

In this section, we compare four chunking strategies on the same text. We keep `chunk_size` and `chunk_overlap` fixed so the comparison is fair

For each strategy, the code follows the same flow: create a splitter, split `docs_merged`, then print basic statistics (number of chunks, total characters, and average chunk length)

In [11]:
chunk_size = 450
chunk_overlap = 30

First, we define a helper function that summarizes chunk outputs

`chunking_stats` takes a list of chunks and returns: total chunks, total characters, and average chunk length. This makes it easy to compare splitters using the same metrics

In [12]:
def chunking_stats(chunks: list[langchain_core.documents.base.Document]):
    total_documents = len(chunks)
    if total_documents < 1:
        return 0, 0, 0
    total_length = sum(len(chunk.page_content) for chunk in chunks)
    avg_length = total_length / total_documents
    return total_documents, total_length, avg_length

### Simple Chunking

Simple chunking splits text into fixed-size token windows with overlap. It is fast and predictable, but it may cut ideas in the middle.

In the next code cell, we create a `TokenTextSplitter`, split `docs_merged`, and print chunk statistics for this baseline approach.

In [13]:
text_splitter_fixed = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
documents_fixed_split = text_splitter_fixed.split_documents(docs_merged)
res = chunking_stats(documents_fixed_split)
print(f"Number of chunks {res[0]}, total characters {res[1]:,}, average characters {res[2]:.2f}")

Number of chunks 1, total characters 450, average characters 450.00


### Recursive Chunking

Recursive chunking tries larger boundaries first (for example paragraphs), then falls back to smaller boundaries only when needed. This usually keeps text units more natural

In the next code cell, we build a `RecursiveCharacterTextSplitter`, split the same input text, and print the same statistics for side-by-side comparison.

In [14]:
text_splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=30
)

documents_recursive_split = text_splitter_recursive.split_documents(docs_merged)

print(chunking_stats(documents_recursive_split))

(2, 446, 223.0)


### Semantic Chunking

Semantic chunking uses embeddings to split where meaning changes, instead of splitting only by size. This can improve retrieval quality, but chunk sizes are less uniform.

`breakpoint_threshold_type` controls how split points are detected from semantic distance values between neighboring text segments.
With `percentile`, the splitter cuts where distances are above a chosen percentile. Lower thresholds usually create more, smaller chunks; higher thresholds create fewer, larger chunks.
You can tune this behavior with `breakpoint_threshold_amount` to match your document style and retrieval needs.

In the next cells, we configure an embedding model, build a `SemanticChunker`, split the document, and print chunk statistics.

In [15]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

Now create the semantic splitter:

In [16]:
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Split the document:

In [17]:
document_semantic_split = semantic_splitter.split_documents(docs_merged)

print(chunking_stats(document_semantic_split))

(2, 445, 222.5)


### LLM chunking

LLM chunking asks a language model to produce self-contained chunks based on meaning and structure. It can create highly coherent chunks, but it is slower and more expensive at indexing time.

In the next cells, we configure a chat model, send a chunking prompt, parse the model response into a list of chunks, convert each chunk into a `Document`, and print summary statistics

In [18]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434"
)

In [19]:
chunking_prompt = ChatPromptTemplate.from_template("""
You are an expert in medicine.

Split the following document into meaningful chunks.

Rules:
- Each chunk should contain one complete medicine topic.
- Return ONLY a valid JSON array.
- Do NOT write explanations.
- Do NOT write "Here is the output".
- Output format:

[
  "chunk1",
  "chunk2",
  "chunk3"
]

DOCUMENT:
{document}
""")

Run the chain:

In [20]:
chunking_chain = chunking_prompt | llm

llm_response = chunking_chain.invoke({
    "document": docs_merged[0].page_content
})

llm_split = eval(llm_response.content)

documents_llm_split = [
    langchain_core.documents.Document(page_content=chunk)
    for chunk in llm_split
]

print(chunking_stats(documents_llm_split))

(2, 38, 19.0)


### Install FAISS Vector 

In [21]:
!pip install faiss-cpu

check the FAISS Vector installed or not:

In [22]:
from langchain_community.vectorstores import FAISS

print("FAISS imported successfully")

FAISS imported successfully


In [23]:
print(type(embeddings))
print(type(documents_fixed_split))
print(type(documents_recursive_split))
print(type(document_semantic_split))

<class 'langchain_ollama.embeddings.OllamaEmbeddings'>
<class 'list'>
<class 'list'>
<class 'list'>


## Build the FAISS Vector Database


In [24]:
vectorstoredb_fixed = FAISS.from_documents(
    documents_fixed_split,
    embeddings
)

vectorstoredb_recursive = FAISS.from_documents(
    documents_recursive_split,
    embeddings
)

vectorstoredb_semantic = FAISS.from_documents(
    document_semantic_split,
    embeddings
)

print("✅ All vector databases created successfully!")

✅ All vector databases created successfully!


Now let's query the four vector databases. Each one uses content split with a different chunking technique.

Ask a Medicine Question

In [25]:
query = "What are the uses of Paracetamol?"

Search using Fixed Chunking

In [26]:
similarity_result_fixed = vectorstoredb_fixed.similarity_search(query)
print(similarity_result_fixed[0].page_content)


Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day.
Avoid alcohol while taking this medicine.

Medicine: Aspirin

Uses:
Pain relief
Reduces fever
Blood thinner

Dosage:
75–325 mg as prescribed.

Side Effects:
Stomach irritation
Bleeding
Heartburn

Precautions:
Avoid if allergic to aspirin.



Search using Recursive Chunking

In [28]:
similarity_result_recursive = vectorstoredb_recursive.similarity_search(query)

print(similarity_result_recursive[0].page_content)

Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain.

Dosage:
500 mg every 4–6 hours.

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day.
Avoid alcohol while taking this medicine.

Medicine: Aspirin

Uses:
Pain relief
Reduces fever
Blood thinner

Dosage:
75–325 mg as prescribed.


Search using Semantic Chunking

In [29]:
similarity_result_semantic = vectorstoredb_semantic.similarity_search(query)

print(similarity_result_semantic[0].page_content)


Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain. Dosage:
500 mg every 4–6 hours.


Retrieve Multiple Results

In [30]:
results = vectorstoredb_semantic.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results):
    print(f"\n========== Result {i+1} ==========\n")
    print(doc.page_content)


========== Result 1 ==========


Medicine: Paracetamol

Uses:
Relieves fever and mild to moderate pain. Dosage:
500 mg every 4–6 hours.

========== Result 2 ==========

Side Effects:
Nausea
Vomiting
Rash
Liver damage in overdose

Precautions:
Do not exceed 4 grams per day. Avoid alcohol while taking this medicine. Medicine: Aspirin

Uses:
Pain relief
Reduces fever
Blood thinner

Dosage:
75–325 mg as prescribed. Side Effects:
Stomach irritation
Bleeding
Heartburn

Precautions:
Avoid if allergic to aspirin. 


### Test with Different Medicine Questions

In [32]:
query = "What are the side effects of Paracetamol?"

In [33]:
query = "What is Aspirin used for?"

In [34]:
query = "What is the dosage of Cetirizine?"

In [35]:
query = "What precautions should diabetic patients follow?"

### (Optional): Save the FAISS Database

Instead of recreating the vector database every time, save it:

In [36]:
vectorstoredb_semantic.save_local("medicine_faiss")

Later, you can load it with:

In [37]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "medicine_faiss",
    embeddings,
    allow_dangerous_deserialization=True
)

## Exercises for the Reader

- Change `chunk_size` and `chunk_overlap`
- Explore other `breakpoint_threshold_type` in the `SemanticChunker`, e.g., `standard_deviation` or `interquartile`
- Try a different VectorDB such as [Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) by replacing `FAISS` with `Chroma`
  - Do you note any difference in the quality of results?


## Conclusions

This notebook showed how chunking strategy directly affects what gets indexed and what gets retrieved in a RAG pipeline.

By comparing simple, recursive, semantic, and LLM-based chunking, you can see trade-offs between chunk structure, coherence, and retrieval behavior.

The interactive dashboards make this comparison practical: one helps inspect chunk content across strategies, and the other helps test a query and review retrieval results by vector database.

In practice, there is no universal best strategy—effective chunking depends on your document style, query patterns, and quality targets.

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT